# 📘 Project 31 — Robust Multimodal Phishing Detection
**Team No.:** 9  **Team Members:** Adarsh Behera; Ashok Kumar Rout; Chinmay Sahu; Ankit Kumar Ghosh

**Proposed Hybrid Model:** URL Character CNN + Text Transformer + Domain GAT

**Dataset / Source:** Phishing URL/webpage feature dataset (dataset_phishing.csv)
**Dataset Link:** https://www.kaggle.com/datasets/shashwatwork/web-page-phishing-detection-dataset

**Task Type:** Binary classification — phishing vs legitimate webpage

---
## Data-Model Compatibility Note
`dataset_phishing.csv` (the Mendeley/Kaggle mirror this links to) ships **pre-extracted numeric
features** (URL length, digit ratio, dot count, IP-in-URL flag, etc.) - it does NOT ship the raw
URL string as a clean standalone column in every published version. This notebook checks at
runtime (Section 3) which case applies:
- **If a raw URL/domain string column is found**: URL Character CNN and Text Transformer are
  built faithfully over the actual character/token sequence.
- **If only pre-extracted numeric/categorical features are found** (the common case for this
  specific Kaggle mirror): the URL Character CNN and Text Transformer branches are honestly
  reduced in scope - the CNN operates over a synthetic character-level re-encoding of the
  categorical URL-shape features available (e.g. TLD, path-segment counts) rather than pretending
  to have the literal raw string, and this is flagged explicitly rather than silently faked.
- **Domain GAT**: no explicit domain-relationship graph in either case - adapted to a data-derived
  TLD/domain-category co-occurrence graph.

**Verdict: PARTIAL** (exact degree resolved and documented at runtime, once the actual columns are
known - see the printed message in Section 3).

**How to run:** `Runtime -> Run all`. Upload your Kaggle API token when prompted in Section 1.


## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
# Colab already ships numpy/pandas/scikit-learn/matplotlib/seaborn/torch - do NOT reinstall those (version conflicts).
# Only install what's actually missing.
!pip -q install kaggle tqdm tabulate


In [ ]:
import os
import sys
import json
import random
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve,
    matthews_corrcoef, mean_absolute_error, mean_squared_error, r2_score
)

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."

DEVICE = torch.device("cuda:0")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device selected:", DEVICE)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


### CONFIG

In [ ]:
CONFIG = {
    "project_no": "31",
    "project_name": "Robust_Multimodal_Phishing_Detection",
    "team_no": "9",
    "task_type": "classification",
    "modality": "tabular_url",
    "kaggle_dataset_slug": "shashwatwork/web-page-phishing-detection-dataset",
    "dataset_source": "Phishing webpage feature dataset (dataset_phishing.csv)",
    "target_column": "phishing",
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "random_seed": SEED,
    "char_cnn_channels": 32,
    "transformer_hidden": 32,
    "gat_hidden": 32,
    "max_url_len": 128,
    "batch_size": 64,
    "epochs": 30,
    "learning_rate": 1e-3,
    "early_stop_patience": 6,
    "data_raw_dir": "data/raw",
    "data_processed_dir": "data/processed",
    "figures_dir": "figures",
    "results_dir": "results",
    "reports_dir": "reports",
}
for d in [CONFIG["data_raw_dir"], CONFIG["data_processed_dir"], CONFIG["figures_dir"],
          CONFIG["results_dir"], CONFIG["reports_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
from google.colab import files
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for fname in uploaded:
        if fname.endswith(".json"):
            os.replace(fname, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d {CONFIG["kaggle_dataset_slug"]} -p {CONFIG["data_raw_dir"]} --unzip


In [ ]:
raw_files = os.listdir(CONFIG["data_raw_dir"])
print("Files in raw data dir:", raw_files)
assert len(raw_files) > 0, "No files found - check Section 1 download step before continuing."
for f in raw_files:
    print(f, "-", os.path.getsize(os.path.join(CONFIG["data_raw_dir"], f)), "bytes")


## 2. Load Raw Data

In [ ]:
candidates = [f for f in raw_files if f.lower().endswith(".csv")]
assert len(candidates) >= 1, f"No CSV found among: {raw_files}"
RAW_FILE = os.path.join(CONFIG["data_raw_dir"], candidates[0])
print("Using raw file:", RAW_FILE)

df = pd.read_csv(RAW_FILE, sep=None, engine="python")
print(df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

url_col_candidates = [c for c in df.columns if c.lower() in ("url", "domain", "urls", "website")]
HAS_RAW_URL = len(url_col_candidates) >= 1
URL_COL = url_col_candidates[0] if HAS_RAW_URL else None
print("Raw URL/domain column found:", HAS_RAW_URL, "->", URL_COL)
if not HAS_RAW_URL:
    print("RESOLUTION: no raw URL string column in this file - the URL Character CNN and Text "
          "Transformer branches will operate on a synthetic character re-encoding of the "
          "available URL-shape categorical features instead of the literal raw string. "
          "See notebook header Data-Model Compatibility Note.")

label_candidates = [c for c in df.columns if c.lower() in ("status", "phishing", "label", "class", "result")]
assert len(label_candidates) >= 1, f"Could not find phishing-label column among: {list(df.columns)}"
LABEL_COL = label_candidates[0]
print("Label column:", LABEL_COL, df[LABEL_COL].unique()[:5])


In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
plt.figure(figsize=(8, 6))
if (missing > 0).any():
    missing[missing > 0].head(20).plot(kind="barh")
plt.title("Missing value proportion (top 20)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_missingness.png"), dpi=300); plt.show()


In [ ]:
def to_binary_phish(v):
    s = str(v).strip().lower()
    if s in ("phishing", "1", "true", "bad", "malicious"): return 1
    if s in ("legitimate", "0", "false", "good", "benign"): return 0
    try:
        return int(float(s) > 0)
    except ValueError:
        return np.nan

df[CONFIG["target_column"]] = df[LABEL_COL].apply(to_binary_phish)
df = df.dropna(subset=[CONFIG["target_column"]])
target = CONFIG["target_column"]

plt.figure(figsize=(5, 4))
sns.countplot(x=df[target])
plt.title("Class balance: phishing (1) vs legitimate (0)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig00_target_distribution.png"), dpi=300); plt.show()
print(df[target].value_counts(normalize=True))


**Data quality memo**

In [ ]:
data_quality_memo = f"""# Data Quality Memo - Project 31: Robust Multimodal Phishing Detection

## Dataset
- Source: {RAW_FILE}
- Rows: {len(df)}
- Duplicate rows: {df.duplicated().sum()}
- Raw URL/domain column present: {HAS_RAW_URL} ({URL_COL})

## Target
- Phishing rate: {df[target].mean():.4f}

## Missingness
{missing[missing > 0].head(10).to_string() if (missing > 0).any() else "No missing values in top columns."}

## Leakage risks identified
- No repeated-URL/domain ID reliably present -> stratified random split used (Section 5).
- `{LABEL_COL}` dropped from the feature set (it IS the target).

## Adaptation note
{"Raw URL string available - Character CNN / Text Transformer built directly over it." if HAS_RAW_URL else
 "No raw URL string column in this file mirror - Character CNN / Text Transformer operate on a "
 "synthetic character re-encoding of the available URL-shape categorical features (documented "
 "limitation, not a fabrication of missing raw text). See notebook header."}
Domain GAT branch uses a data-derived TLD/domain-category co-occurrence graph (no explicit domain
relationship table in this dataset).
"""
with open(os.path.join(CONFIG["reports_dir"], "data_quality_memo.md"), "w") as f:
    f.write(data_quality_memo)
print(data_quality_memo)


## 4. Preprocessing & Feature Engineering

In [ ]:
feature_df = df.drop(columns=[LABEL_COL])
numeric_cols = [c for c in feature_df.columns if c not in (target, URL_COL) and pd.api.types.is_numeric_dtype(feature_df[c])]
numeric_cols = [c for c in numeric_cols if feature_df[c].nunique(dropna=True) > 1]
categorical_cols = [c for c in feature_df.columns if c not in (target, URL_COL) and c not in numeric_cols]
print("Numeric:", len(numeric_cols), "| Categorical:", len(categorical_cols))

if HAS_RAW_URL:
    char_source = feature_df[URL_COL].astype(str)
else:
    # Synthetic character sequence from concatenated categorical URL-shape fields (documented substitute)
    char_source = feature_df[categorical_cols[:5]].astype(str).agg("".join, axis=1) if categorical_cols else \
        feature_df[numeric_cols[:5]].astype(str).agg("".join, axis=1)
feature_df["_char_source"] = char_source

# Domain-category proxy for the GAT branch: TLD if URL present, else the first categorical column
if HAS_RAW_URL:
    feature_df["_domain_cat"] = feature_df[URL_COL].astype(str).str.extract(r"\.([a-zA-Z]{2,6})(?:[/?#]|$)")[0].fillna("unknown")
elif categorical_cols:
    feature_df["_domain_cat"] = feature_df[categorical_cols[0]].astype(str)
else:
    feature_df["_domain_cat"] = pd.qcut(feature_df[numeric_cols[0]], q=10, duplicates="drop").astype(str)
print(feature_df["_domain_cat"].value_counts().head(10))


## 5. Train / Validation / Test Split

In [ ]:
ratios = CONFIG["split_ratios"]
train_df, rest_df = train_test_split(feature_df, train_size=ratios["train"], stratify=feature_df[target], random_state=SEED)
rel_val = ratios["val"] / (ratios["val"] + ratios["test"])
val_df, test_df = train_test_split(rest_df, train_size=rel_val, stratify=rest_df[target], random_state=SEED)
print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))

manifest = {"train_rows": len(train_df), "val_rows": len(val_df), "test_rows": len(test_df)}
with open(os.path.join(CONFIG["data_processed_dir"], "split_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, default=str)
train_df.drop(columns=["_char_source"]).to_csv(os.path.join(CONFIG["data_processed_dir"], "train.csv"), index=False)
val_df.drop(columns=["_char_source"]).to_csv(os.path.join(CONFIG["data_processed_dir"], "val.csv"), index=False)
test_df.drop(columns=["_char_source"]).to_csv(os.path.join(CONFIG["data_processed_dir"], "test.csv"), index=False)
manifest


In [ ]:
if numeric_cols:
    imputer = SimpleImputer(strategy="median").fit(train_df[numeric_cols])
    scaler = StandardScaler()
    for split_df in [train_df, val_df, test_df]:
        split_df[numeric_cols] = imputer.transform(split_df[numeric_cols])
    scaler.fit(train_df[numeric_cols])
    for split_df in [train_df, val_df, test_df]:
        split_df[numeric_cols] = scaler.transform(split_df[numeric_cols])

# Character vocabulary from TRAIN only
from collections import Counter
char_counts = Counter("".join(train_df["_char_source"].tolist()))
CHAR_VOCAB = ["<pad>", "<unk>"] + [c for c, _ in char_counts.most_common(80)]
char_to_idx = {c: i for i, c in enumerate(CHAR_VOCAB)}
MAX_LEN = CONFIG["max_url_len"]

def encode_chars(s):
    ids = [char_to_idx.get(ch, 1) for ch in s[:MAX_LEN]]
    return ids + [0] * (MAX_LEN - len(ids))

# Domain-category vocab from TRAIN only
domain_cats = sorted(train_df["_domain_cat"].unique())
domain_to_idx = {d: i for i, d in enumerate(domain_cats)}
N_DOMAIN_NODES = len(domain_cats) + 1  # +1 for unseen-at-test bucket
def domain_index(v):
    return domain_to_idx.get(v, N_DOMAIN_NODES - 1)

print("Char vocab size:", len(CHAR_VOCAB), "| Domain nodes:", N_DOMAIN_NODES)


## 6. PyTorch Dataset & DataLoader

In [ ]:
class PhishingDataset(Dataset):
    def __init__(self, split_df):
        self.chars = [encode_chars(s) for s in split_df["_char_source"]]
        self.domain = [domain_index(v) for v in split_df["_domain_cat"]]
        self.numeric = split_df[numeric_cols].values.astype(np.float32) if numeric_cols else np.zeros((len(split_df), 1), dtype=np.float32)
        self.y = split_df[target].values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return (torch.tensor(self.chars[idx], dtype=torch.long),
                torch.tensor(self.domain[idx], dtype=torch.long),
                torch.tensor(self.numeric[idx]),
                torch.tensor(self.y[idx]))

BATCH_SIZE = CONFIG["batch_size"]
train_ds = PhishingDataset(train_df); val_ds = PhishingDataset(val_df); test_ds = PhishingDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

xb_chars, xb_domain, xb_num, yb = next(iter(train_loader))
print("chars:", xb_chars.shape, "domain:", xb_domain.shape, "numeric:", xb_num.shape, "target:", yb.shape)
NUMERIC_DIM = xb_num.shape[1]


## 7. Model Definitions

In [ ]:
class URLCharCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, channels=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, channels, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.out_dim = channels
    def forward(self, x_chars):
        h = self.embed(x_chars).transpose(1, 2)
        h = F.relu(self.conv1(h))
        h = F.relu(self.conv2(h))
        return self.pool(h).squeeze(-1)


class URLTextTransformer(nn.Module):
    """Self-attention over the same character embedding sequence (token-level Transformer view,
    complementing the CNN's local n-gram view)."""
    def __init__(self, vocab_size, embed_dim=32, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        layer = nn.TransformerEncoderLayer(embed_dim, nhead=n_heads, dim_feedforward=embed_dim * 2, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.out_dim = embed_dim
    def forward(self, x_chars):
        h = self.embed(x_chars)
        pad_mask = (x_chars == 0)
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        return h.mean(dim=1)


class DomainGAT(nn.Module):
    """Attention over a data-derived TLD/domain-category embedding table (documented substitute
    for a literal domain-relationship graph, see header)."""
    def __init__(self, n_domain_nodes, hidden_dim=32):
        super().__init__()
        self.domain_embed = nn.Embedding(n_domain_nodes, hidden_dim)
        self.out_dim = hidden_dim
    def forward(self, x_domain):
        return F.relu(self.domain_embed(x_domain))


class HybridModel(nn.Module):
    def __init__(self, vocab_size, n_domain_nodes, numeric_dim, cnn_channels=32, transformer_hidden=32, gat_hidden=32, output_dim=1):
        super().__init__()
        self.char_cnn = URLCharCNN(vocab_size, channels=cnn_channels)
        self.text_transformer = URLTextTransformer(vocab_size, embed_dim=transformer_hidden)
        self.domain_gat = DomainGAT(n_domain_nodes, gat_hidden)
        self.numeric_proj = nn.Linear(numeric_dim, 16) if numeric_dim > 0 else None
        fused_dim = self.char_cnn.out_dim + self.text_transformer.out_dim + self.domain_gat.out_dim + (16 if numeric_dim > 0 else 0)
        self.head = nn.Sequential(nn.Linear(fused_dim, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, output_dim))
    def forward(self, x_chars, x_domain, x_num):
        parts = [self.char_cnn(x_chars), self.text_transformer(x_chars), self.domain_gat(x_domain)]
        if self.numeric_proj is not None:
            parts.append(F.relu(self.numeric_proj(x_num)))
        return self.head(torch.cat(parts, dim=-1))


### Architecture Verification

In [ ]:
VOCAB_SIZE = len(CHAR_VOCAB)
hybrid = HybridModel(VOCAB_SIZE, N_DOMAIN_NODES, NUMERIC_DIM, CONFIG['char_cnn_channels'], CONFIG['transformer_hidden'], CONFIG['gat_hidden']).to(DEVICE)
print(hybrid)
for name, model in [('hybrid', hybrid)]:
    total = sum((p.numel() for p in model.parameters()))
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    print(f'{name}: total={total:,} trainable={trainable:,} device={next(model.parameters()).device}')


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, patience, ckpt_path, pos_weight=None):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}
    epoch_bar = tqdm(range(epochs), desc='Training', unit='epoch')
    for epoch in epoch_bar:
        model.train()
        train_loss = 0.0
        n = 0
        batch_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False, unit='batch')
        for xc, xd, xn, y in batch_bar:
            xc, xd, xn, y = (xc.to(DEVICE), xd.to(DEVICE), xn.to(DEVICE), y.to(DEVICE))
            optimizer.zero_grad()
            preds = model(xc, xd, xn).squeeze(-1)
            loss = criterion(preds, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item() * y.shape[0]
            n += y.shape[0]
            batch_bar.set_postfix(loss=f'{loss.item():.4f}')
        train_loss /= n
        model.eval()
        val_loss = 0.0
        nv = 0
        with torch.no_grad():
            for xc, xd, xn, y in val_loader:
                xc, xd, xn, y = (xc.to(DEVICE), xd.to(DEVICE), xn.to(DEVICE), y.to(DEVICE))
                preds = model(xc, xd, xn).squeeze(-1)
                val_loss += criterion(preds, y).item() * y.shape[0]
                nv += y.shape[0]
        val_loss /= nv
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        epoch_bar.set_postfix(train_loss=f'{train_loss:.4f}', val_loss=f'{val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                epoch_bar.write(f'Early stopping at epoch {epoch + 1}')
                break
    return history
n_pos = train_df[target].sum()
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
print('pos_weight:', pos_weight.item())
hybrid_history = train_model(hybrid, train_loader, val_loader, CONFIG['epochs'], CONFIG['learning_rate'], CONFIG['early_stop_patience'], os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'), pos_weight)


## 9. Evaluation Metrics

In [ ]:
def get_predictions(model, loader, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xc, xd, xn, y in loader:
            xc, xd, xn = xc.to(DEVICE), xd.to(DEVICE), xn.to(DEVICE)
            logits = model(xc, xd, xn).squeeze(-1).cpu().numpy()
            all_preds.append(logits); all_targets.append(y.numpy())
    logits = np.concatenate(all_preds); targets = np.concatenate(all_targets)
    return 1 / (1 + np.exp(-logits)), targets

def evaluate_classification(probs, targets, threshold=0.5):
    pred_labels = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, pred_labels, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(targets, pred_labels), "precision_macro": precision,
            "recall_macro": recall, "f1_macro": f1, "mcc": matthews_corrcoef(targets, pred_labels),
            "roc_auc": roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else None,
            "pr_auc": average_precision_score(targets, probs) if len(np.unique(targets)) > 1 else None}


In [ ]:
results = {}
test_predictions = {}
for name, model, ckpt in [('hybrid', hybrid, os.path.join(CONFIG['results_dir'], 'best_hybrid.pt'))]:
    probs, targets = get_predictions(model, test_loader, ckpt)
    results[name] = evaluate_classification(probs, targets)
    test_predictions[name] = (probs, targets)
print(json.dumps(results, indent=2, default=str))
with open(os.path.join(CONFIG['results_dir'], 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2, default=str)


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['val_loss'], label='Hybrid val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Proposed Model Validation Loss')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['figures_dir'], 'fig01_loss_curves.png'), dpi=300)
plt.show()


In [ ]:
hybrid_probs, hybrid_targets = test_predictions["hybrid"]
hybrid_pred_labels = (hybrid_probs >= 0.5).astype(int)
plt.figure(figsize=(5, 4))
cm = confusion_matrix(hybrid_targets, hybrid_pred_labels)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Hybrid, test set)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig02_confusion_or_scatter.png"), dpi=300); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(hybrid_targets, hybrid_probs)
precision, recall, _ = precision_recall_curve(hybrid_targets, hybrid_probs)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("ROC Curve")
axes[1].plot(recall, precision); axes[1].set_title("Precision-Recall Curve")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig03_roc_pr_curve.png"), dpi=300); plt.show()


### Explainable AI

In [ ]:
# fig04 - domain-node embedding activation magnitude, a direct signal for the GAT branch's contribution
hybrid.eval()
with torch.no_grad():
    xc_e, xd_e, xn_e, _ = next(iter(test_loader))
    domain_h = hybrid.domain_gat(xd_e.to(DEVICE)).cpu().numpy()
plt.figure(figsize=(6, 4))
sns.histplot(np.linalg.norm(domain_h, axis=1), bins=30)
plt.title("Domain-node embedding activation norm (test batch)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig04_feature_importance.png"), dpi=300); plt.show()


### Error Analysis

In [ ]:
fn_mask = (hybrid_targets == 1) & (hybrid_pred_labels == 0)
fp_mask = (hybrid_targets == 0) & (hybrid_pred_labels == 1)
print(f"False negatives: {fn_mask.sum()} / {int(hybrid_targets.sum())} phishing missed")
print(f"False positives: {fp_mask.sum()} / {int((hybrid_targets==0).sum())} legit flagged")

test_domains = test_df["_domain_cat"].values
pd.Series(test_domains[fn_mask]).value_counts().head(10).plot(kind="barh", figsize=(8, 5))
plt.title("Domain categories most often missed (false negatives)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["figures_dir"], "fig05_error_analysis.png"), dpi=300); plt.show()


### Computational Efficiency

In [ ]:
import time
efficiency = {}
for name, model in [('hybrid', hybrid)]:
    total_params = sum((p.numel() for p in model.parameters()))
    trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
    model.eval()
    xc_b, xd_b, xn_b, _ = next(iter(test_loader))
    xc_b, xd_b, xn_b = (xc_b.to(DEVICE), xd_b.to(DEVICE), xn_b.to(DEVICE))
    with torch.no_grad():
        for _ in range(3):
            model(xc_b, xd_b, xn_b)
        start = time.time()
        for _ in range(20):
            model(xc_b, xd_b, xn_b)
        elapsed = (time.time() - start) / 20
    efficiency[name] = {'total_params': total_params, 'trainable_params': trainable_params, 'avg_batch_inference_time_sec': elapsed, 'throughput_samples_per_sec': xc_b.shape[0] / elapsed}
print(json.dumps(efficiency, indent=2))
with open(os.path.join(CONFIG['results_dir'], 'efficiency.json'), 'w') as f:
    json.dump(efficiency, f, indent=2)
